In [1]:
import numpy as np
import pandas as pd
TARGETS = ["Theta"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [2]:
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        for col in [col_r2, col_mse]:
            if col in results.columns:
                results[col] = (
                    results[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )

In [3]:
best_models_tables = {}
best_only = []

summary_all = []     # estatísticas com todos
summary_topN = []   # estatísticas com top 10

N = 10

import pandas as pd

def summarize_models(results, TARGETS, SETS, N=10, output="Resumo_Estatisticas.xlsx"):
    
    best_models_tables = {}
    best_only = []
    summary_all = []
    summary_top10 = []

    for target in TARGETS:
        for dataset in SETS:

            col_r2 = f"R2_{dataset}_{target}"
            col_mse = f"MSE_{dataset}_{target}"

            if col_r2 not in results.columns:
                continue

            table = results.copy()

            table = table.sort_values(by=col_r2, ascending=False)

            best_models_tables[f"{dataset}_{target}"] = table

            # 🔹 melhor modelo
            best = table.iloc[0]

            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": best["model"],
                "Neurons": best["Neurons"],
                "Best_R2": best[col_r2]
            })

            # 🔹 estatísticas de todos
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })

            # 🔹 estatísticas top N
            topN = table.head(N)

            summary_topN.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": topN[col_r2].mean(),
                "std_r2": topN[col_r2].std(),
                "mean_mse": topN[col_mse].mean(),
                "std_mse": topN[col_mse].std()
            })

    df_summary_all = pd.DataFrame(summary_all)
    df_summary_topN = pd.DataFrame(summary_topN)
    best_only_df = pd.DataFrame(best_only)
    # exportar
    with pd.ExcelWriter(output) as writer:
        df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
        df_summary_topN.to_excel(writer, sheet_name="Top_Modelos", index=False)
        best_only_df.to_excel(writer, sheet_name="Melhor_Modelo", index=False)

    return best_models_tables, df_summary_all, df_summary_topN, best_only_df

In [4]:
tables, summary_all, summary_top10, best_models = summarize_models(
    results, TARGETS, SETS
)

In [6]:
summary_top10["bestR2"] = best_models["Best_R2"]
summary_top10["Neurons"] = best_models["Neurons"]
summary_top10["Best_Model"] = best_models["Best_Model"]
display(summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons,Best_Model
0,Train_1,Theta,0.905482,0.010694,0.021575,0.002441,0.927976,"[32, 16, 8]",model_32_16_8_Ld0.7_Lp0.3_r0.01_seed0
1,Train_2,Theta,0.860993,0.004174,0.031736,0.000953,0.868292,"[264, 128]",model_264_128_Ld0.7_Lp0.3_r0.5_seed0
2,Val,Theta,0.548981,0.123466,0.147609,0.040408,0.752385,"[16, 8, 4]",model_264_Ld0.3_Lp0.7_r0.9_seed1
3,Test_1,Theta,0.887523,0.018836,0.025838,0.004327,0.920384,"[32, 16, 8]",model_32_16_8_Ld0.3_Lp0.7_r0.01_seed0
4,Test_2,Theta,0.887611,0.004757,0.026654,0.001128,0.897153,"[16, 8, 4]",model_16_8_4_Ld0.3_Lp0.7_r0.5_seed2
5,Test_3,Theta,0.745633,0.021911,0.083622,0.007203,0.787701,"[16, 8]",model_16_8_Ld0.3_Lp0.7_r0.01_seed0


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted